<a href="https://colab.research.google.com/github/ajaykumar080286/feature_engineering/blob/main/14_titanic_without_using_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [80]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder,OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score

In [81]:
df=pd.read_csv("https://raw.githubusercontent.com/ajaykumar080286/feature_engineering/main/train.csv")

In [82]:
df.head(2)

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C


In [83]:
df.drop(columns=["PassengerId","Name","Cabin","Ticket"], inplace=True)

In [84]:
df.head(3)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S


In [85]:
X=df.iloc[:,1:8]
y=df.iloc[:,0]

In [86]:
X_train,X_test, y_train, y_test= train_test_split(X,y, test_size=0.2, random_state=42)

In [87]:
X_train.isnull().sum()

,0
Pclass,0
Sex,0
Age,140
SibSp,0
Parch,0
Fare,0
Embarked,2


In [88]:
X_train.head(1)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5,S


In [89]:
simpleImp_age=SimpleImputer()
simpleImp_embarked=SimpleImputer(strategy="most_frequent")

In [90]:
X_train_age=simpleImp_age.fit_transform(X_train[["Age"]])
X_test_age=simpleImp_age.transform(X_test[["Age"]])

In [91]:
X_train_embarked=simpleImp_embarked.fit_transform(X_train[["Embarked"]])
X_test_embarked=simpleImp_embarked.transform(X_test[["Embarked"]])

In [92]:
X_train_age.shape

(712, 1)

In [93]:
ohe=OneHotEncoder(sparse_output=False, handle_unknown="ignore")
ohe_emb=OneHotEncoder(sparse_output=False, handle_unknown="ignore")

In [94]:
X_train_sex=ohe.fit_transform(X_train[["Sex"]])
X_test_sex=ohe.transform(X_test[["Sex"]])


X_train_emb=ohe_emb.fit_transform(X_train_embarked)
X_test_emb=ohe_emb.transform(X_test_embarked)


In [95]:
X_train_emb.shape

(712, 3)

In [96]:
X_train_ren=X_train.drop(columns=["Sex","Age","Embarked"]).values
X_test_ren=X_test.drop(columns=["Sex","Age","Embarked"]).values

In [97]:
X_train_ren.shape

(712, 4)

In [98]:
X_train_trans=np.concatenate((X_train_ren,X_train_sex,X_train_age, X_train_emb),axis=1)
X_test_trans=np.concatenate((X_test_ren,X_test_sex,X_test_age, X_test_emb),axis=1)

In [99]:
X_train_trans.shape

(712, 10)

In [100]:
from sklearn.tree import DecisionTreeClassifier

In [101]:
dt=DecisionTreeClassifier()

In [102]:
dt.fit(X_train_trans, y_train)

DecisionTreeClassifier()

In [103]:
y_pred=dt.predict(X_test_trans)

In [104]:
accuracy_score(y_pred,y_test)

0.7821229050279329

In [105]:
import pickle

In [106]:
pickle.dump(dt,open("models/dt.pkl","wb"))

In [107]:
pickle.dump(ohe,open("models/ohe.pkl","wb"))

In [108]:
pickle.dump(ohe_emb,open("models/ohe_emb.pkl","wb"))

**Preiction Without Pipe Line**

In [109]:
ohe_sex = pickle.load(open('models/ohe.pkl','rb'))
ohe_embarked = pickle.load(open('models/ohe_emb.pkl','rb'))
clf = pickle.load(open('models/dt.pkl','rb'))

In [115]:
df.head(1)

,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.25,S


In [123]:
new_df=pd.DataFrame([2, 'male', 31.0, 0, 0, 10.5, 'S'])

In [124]:
test_input=new_df.values.reshape(1,7)

In [125]:
test_input

array([[2, 'male', 31.0, 0, 0, 10.5, 'S']], dtype=object)

In [133]:
ohe_sex_1=ohe_sex.transform(test_input[:,1].reshape(1, 1))

In [135]:
ohe_emb_1=ohe_embarked.transform(test_input[:,6].reshape(1, 1))

In [137]:
ohe_emb_1.shape

(1, 3)

In [138]:
ohe_sex_1.shape

(1, 2)

In [141]:
age=test_input[:,2].reshape(1,1)

In [145]:
X_rem=test_input[:,[0,3,4,5]]

In [146]:
X_final_data=np.concatenate((X_rem,ohe_sex_1,age, ohe_emb_1),axis=1)


In [147]:
X_final_data.shape

(1, 10)

In [148]:
dt.predict(X_final_data)

array([0])